In [ ]:
 #2 – Mapa a k ní statistika s 2x výběrem (kontinenty) a celkem případy, celkem smrt, celkem očkování, celkem testy
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

# Načtení dat (poslední den datasetu)
df = pd.read_csv(r"D:\CodersLab\project_1_python.csv")
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df_last = df[df["date"] == df["date"].max()].copy()  # potřebuju poslední

# Dash aplikace
app = Dash(__name__)

# Layout
app.layout = html.Div([
    html.H1("COVID-19 Map Dashboard"),

    html.Label("Select a continent:"),
    dcc.Dropdown(
        id="continent-filter",
        options=[{"label": c, "value": c} for c in df_last["continent"].dropna().unique()],
        value="Europe"
    ),

    html.Label("Select a metric:"),
    dcc.Dropdown(
        id="metric-filter",
        options=[
            {"label": "Total cases", "value": "total_cases"},
            {"label": "Total deaths", "value": "total_deaths"},
            {"label": "Total tests", "value": "total_tests"},
            {"label": "Total vaccinations", "value": "total_vaccinations"},
            {"label": "Total vaccinated people", "value": "people_vaccinated"}
        ],
        value="total_cases"
    ),

    dcc.Graph(id="map-graph")
])
@app.callback(
    Output("map-graph", "figure"),
    [Input("continent-filter", "value"),
     Input("metric-filter", "value")]
)
def update_map(continent, metric):
    dff = df_last[df_last["continent"] == continent].copy()

    # Odstranit řádky, kde je metrika NaN nebo inf
    dff = dff[dff[metric].notna() & (dff[metric] != float("inf"))]

    fig = px.scatter_mapbox(
    dff,
    lat="latitude",
    lon="longitude",
    size=metric,
    hover_name="location",
    hover_data={metric: True, "latitude": True, "longitude": True},
    size_max=50,
    zoom=1,
    mapbox_style="carto-darkmatter",
    title=f"COVID-19 - {metric} in {continent}"
)

    return fig



# Spuštění aplikace
if __name__ == "__main__":
    app.run(debug=True)
